In [0]:
# Synthetic Pharos CDM 2.0.1 + MOSTLY AI hybrid dataset generator
#
# Generates all 17 entity tables and 11 reference tables for breast, lung,
# and pancreas. MOSTLY AI Local runs inside Databricks and learns breast joint values
# and person-linked sequences from the read-only source. Missing model tables,
# lung, and pancreas use the CDM-valid fallback. Identifiers, foreign keys,
# timelines, and cross-table flags are always constructed.
# Leave source_schema blank for pure-spec generation.
#
# Fixed default seed: 20260818. Target write scope is restricted to 8_dev.*.
# Share this notebook with the three synth_pharos_*.py helper files beside it.


In [0]:

# GPU runtime dependencies. The cu128 wheel is compatible with the 12.9 driver
# observed on Small GPU 2/3; MOSTLY AI 6.1.1 requires Torch 2.11.x.
%pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu128
%pip install 'mostlyai[local]==6.1.1'
%restart_python


In [0]:

# Point this at a Databricks directory containing pharos_cdm/__init__.py.
# Leave it blank when pharos_cdm is already installed/importable.
dbutils.widgets.text("cdm_root", "")

import importlib
import sys
from pathlib import Path

CDM_ROOT = dbutils.widgets.get("cdm_root").strip()

def _add_cdm_root(root):
    path = Path(root).resolve()
    if not (path / "pharos_cdm" / "__init__.py").is_file():
        raise FileNotFoundError(
            f"{path} does not contain pharos_cdm/__init__.py; set cdm_root correctly"
        )
    value = str(path)
    if value not in sys.path:
        sys.path.insert(0, value)
    return path

if CDM_ROOT:
    CDM_PATH = _add_cdm_root(CDM_ROOT)
else:
    CDM_PATH = None

importlib.invalidate_caches()
import pharos_cdm
from pharos_cdm.validation.registry import available_tables

assert len(available_tables()) == 17, available_tables()
CODE_SOURCE_INFO = {
    "layout": "relative Databricks %run",
    "files": [
        "Synth_Pharos_CDM2.py",
        "synth_pharos_support.py",
        "synth_pharos_mostly.py",
        "synth_pharos_generate.py",
    ],
}
CDM_SOURCE_INFO = {
    "requested_root": str(CDM_PATH) if CDM_PATH else None,
    "module_file": str(Path(pharos_cdm.__file__).resolve()),
    "version": getattr(pharos_cdm, "__version__", None) or "2.0.1",
}
print("code source:", CODE_SOURCE_INFO)
print("CDM source:", CDM_SOURCE_INFO)
print("CDM tables:", sorted(available_tables()))


In [0]:

%run ./synth_pharos_support


In [0]:

%run ./synth_pharos_mostly


In [0]:

%run ./synth_pharos_generate


In [0]:

dbutils.widgets.dropdown("mode", "fixture", ["fixture", "full"])
dbutils.widgets.text("seed", "20260818")
dbutils.widgets.text("breast_persons", "600")
dbutils.widgets.text("lung_persons", "250")
dbutils.widgets.text("pancreas_persons", "150")
dbutils.widgets.text("source_schema", "8_dev.silver")
dbutils.widgets.text("target", "8_dev.synth_pharos")
dbutils.widgets.dropdown("mostly_mode", "train", ["train", "reuse", "off"])
dbutils.widgets.text("mostly_train_persons", "2000")
dbutils.widgets.text("mostly_tables", ",".join(DEFAULT_TRAIN_TABLES))
dbutils.widgets.text("mostly_max_training_time", "8")
dbutils.widgets.text("mostly_max_epochs", "15")
dbutils.widgets.dropdown("mostly_allow_cpu", "false", ["false", "true"])
MODE = dbutils.widgets.get("mode")
SEED = int(dbutils.widgets.get("seed"))
SOURCE = dbutils.widgets.get("source_schema").strip()
TARGET = dbutils.widgets.get("target").strip()
MOSTLY_MODE = dbutils.widgets.get("mostly_mode").strip()
assert TARGET.startswith("8_dev."), f"write-scope guard: refusing target {TARGET}"
assert len(TARGET.split(".")) == 2, "target must be catalog.schema"
PERSONS = {"breast": 20, "lung": 20, "pancreas": 20} if MODE == "fixture" else {
    "breast": int(dbutils.widgets.get("breast_persons")),
    "lung": int(dbutils.widgets.get("lung_persons")),
    "pancreas": int(dbutils.widgets.get("pancreas_persons")),
}
MOSTLY_TRAIN_PERSONS = int(dbutils.widgets.get("mostly_train_persons"))
MOSTLY_TABLES = tuple(
    table.strip() for table in dbutils.widgets.get("mostly_tables").split(",") if table.strip()
)
MOSTLY_MAX_TRAINING_TIME = float(dbutils.widgets.get("mostly_max_training_time"))
MOSTLY_MAX_EPOCHS = float(dbutils.widgets.get("mostly_max_epochs"))
MOSTLY_ALLOW_CPU = dbutils.widgets.get("mostly_allow_cpu") == "true"
if MODE == "fixture":
    MOSTLY_TRAIN_PERSONS = min(MOSTLY_TRAIN_PERSONS, 400)
    MOSTLY_MAX_TRAINING_TIME = min(MOSTLY_MAX_TRAINING_TIME, 2.0)
    MOSTLY_MAX_EPOCHS = min(MOSTLY_MAX_EPOCHS, 3.0)
print(f"mode={MODE} seed={SEED} persons={PERSONS} source={SOURCE or 'NONE (pure spec)'} "
      f"target={TARGET} mostly_mode={MOSTLY_MODE} mostly_train_persons={MOSTLY_TRAIN_PERSONS} "
      f"mostly_tables={MOSTLY_TABLES} limits=({MOSTLY_MAX_TRAINING_TIME}min,{MOSTLY_MAX_EPOCHS}ep)")


In [0]:

def _quoted_name(name):
    return ".".join(f"`{part.replace('`', '``')}`" for part in name.split("."))

TARGET_SQL = _quoted_name(TARGET)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SQL}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {TARGET_SQL}.`artifacts`")
print(f"provisioned {TARGET} and {TARGET}.artifacts")


In [0]:

REGISTRY = build_registry()
prof = None
PROFILE_COVERAGE = {}
frames, skipped = {}, []
if SOURCE:
    for cdm_table, source_table in SOURCE_TABLE_MAP.items():
        try:
            frames[cdm_table] = spark.table(f"{SOURCE}.{source_table}").toPandas()
        except Exception as exc:
            skipped.append((source_table, str(exc).split(chr(10))[0][:120]))
    prof = build_profile(frames, REGISTRY, min_support=10, source=SOURCE)
    print(f"profiled {len(prof['tables'])} tables from {SOURCE}; skipped={skipped}")
    for table, table_profile in sorted(prof["tables"].items()):
        profiled_columns = sorted(table_profile["columns"])
        candidate_columns = [
            field.name for field in REGISTRY[table].fields
            if not field.meta.is_pk and not field.meta.fk
        ]
        fallback_columns = sorted(set(candidate_columns) - set(profiled_columns))
        groups = sorted({
            group
            for column in table_profile["columns"].values()
            for group in column["by_group"]
        })
        PROFILE_COVERAGE[table] = {
            "profiled_columns": profiled_columns,
            "fallback_columns": fallback_columns,
            "profiled_groups": groups,
        }
        print(f"  {table}: {len(profiled_columns)}/{len(candidate_columns)} columns profiled; groups={groups}")
else:
    print("pure spec mode (no source)")


In [0]:

import gc
from datetime import datetime, timezone
from pathlib import Path

catalog, schema_name = TARGET.split(".")
model_dir = Path(f"/Volumes/{catalog}/{schema_name}/artifacts/models")
report_dir_early = Path(f"/Volumes/{catalog}/{schema_name}/artifacts/reports")
model_dir.mkdir(parents=True, exist_ok=True)
report_dir_early.mkdir(parents=True, exist_ok=True)
MOSTLY_MODEL_PATH = model_dir / "breast_generator.zip"
MOSTLY_DATA, MOSTLY_REPORT = run_mostly(
    frames,
    REGISTRY,
    MostlyRunConfig(
        mode=MOSTLY_MODE,
        seed=SEED,
        breast_persons=PERSONS["breast"],
        training_subjects=MOSTLY_TRAIN_PERSONS,
        tables=MOSTLY_TABLES,
        max_training_time=MOSTLY_MAX_TRAINING_TIME,
        max_epochs=MOSTLY_MAX_EPOCHS,
        allow_cpu=MOSTLY_ALLOW_CPU,
        model_path=MOSTLY_MODEL_PATH,
        work_dir=Path(f"/local_disk0/mostlyai/pharos_{schema_name}_{SEED}"),
        report_dir=report_dir_early,
        generator_name=(
            f"pharos_cdm2_breast_{schema_name}_"
            + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        ),
    ),
)
print("MOSTLY AI report:", MOSTLY_REPORT)

del frames
gc.collect()


In [0]:

ds = generate_dataset(seed=SEED, persons=PERSONS, prof=prof, mostly_data=MOSTLY_DATA)
ROW_COUNTS = {table: int(len(frame)) for table, frame in ds.tables.items()}
print("generated row counts:")
for table, count in sorted(ROW_COUNTS.items()):
    print(f"  {table}: {count}")


In [0]:

validation_report = validate_dataset(ds)
print("validation errors:", validation_report["errors"])
print("per table:", validation_report["by_table"])
if validation_report["errors"]:
    raise RuntimeError(
        "CDM validation failed: " + repr(validation_report["issues"][:25])
    )
print("CDM validation gate passed with zero errors")


In [0]:

from datetime import date, datetime
from decimal import Decimal
import math
import numpy as np
import pandas as pd
from pyspark.sql.types import (
    BooleanType, DateType, DecimalType, DoubleType, FloatType, IntegerType,
    LongType, ShortType, StringType, StructField, StructType, TimestampType,
)

_SPARK_TYPE = {
    "STRING": StringType, "BIGINT": LongType, "INT": IntegerType,
    "SMALLINT": ShortType, "DOUBLE": DoubleType, "FLOAT": FloatType,
    "BOOLEAN": BooleanType, "BOOL": BooleanType, "DATE": DateType,
    "TIMESTAMP": TimestampType,
}

def _spark_type(sql_type):
    kind = (sql_type or "STRING").upper()
    if kind.startswith("DECIMAL"):
        return DecimalType(18, 4)
    if kind not in _SPARK_TYPE:
        raise ValueError(f"unsupported Pharos SQL type: {sql_type}")
    return _SPARK_TYPE[kind]()

def _is_null(value):
    if value is None:
        return True
    try:
        result = pd.isna(value)
        return bool(result) if isinstance(result, (bool, np.bool_)) else False
    except Exception:
        return False

def _coerce(value, sql_type):
    if _is_null(value):
        return None
    if isinstance(value, np.generic):
        value = value.item()
    kind = (sql_type or "STRING").upper()
    if kind == "DATE":
        if isinstance(value, datetime):
            return value.date()
        if isinstance(value, date):
            return value
        return date.fromisoformat(str(value)[:10])
    if kind == "TIMESTAMP":
        if isinstance(value, datetime):
            return value
        return datetime.fromisoformat(str(value))
    if kind in ("BIGINT", "INT", "SMALLINT"):
        return int(value)
    if kind in ("DOUBLE", "FLOAT"):
        return float(value)
    if kind.startswith("DECIMAL"):
        return Decimal(str(value))
    if kind in ("BOOLEAN", "BOOL"):
        return bool(value)
    return str(value)

def _table_name(table):
    return f"{TARGET_SQL}.`{table.replace('`', '``')}`"

def _sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

def _publish_cdm_table(table, frame, spec):
    schema = StructType([
        StructField(field.name, _spark_type(field.meta.sql_type), True)
        for field in spec.fields
    ])
    records = [
        {
            field.name: _coerce(row[field.name], field.meta.sql_type)
            for field in spec.fields
        }
        for row in frame.to_dict("records")
    ]
    spark_frame = spark.createDataFrame(records, schema=schema)
    spark_frame.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        _table_name(table)
    )
    if spec.comment:
        spark.sql(f"COMMENT ON TABLE {_table_name(table)} IS {_sql_literal(spec.comment)}")
    for field in spec.fields:
        if field.meta.description:
            column = field.name.replace("`", "``")
            spark.sql(
                f"ALTER TABLE {_table_name(table)} ALTER COLUMN `{column}` "
                f"COMMENT {_sql_literal(field.meta.description)}"
            )

for table, frame in ds.tables.items():
    _publish_cdm_table(table, frame, ds.registry[table])
    print(f"published {TARGET}.{table}: {len(frame)} rows")

REF_TABLES = build_ref_tables()
for table, frame in REF_TABLES.items():
    fields = []
    for column in frame.columns:
        dtype = BooleanType() if column == "is_regimen" else StringType()
        fields.append(StructField(str(column), dtype, True))
    schema = StructType(fields)
    records = []
    for row in frame.to_dict("records"):
        converted = {}
        for column in frame.columns:
            value = row[column]
            if _is_null(value):
                converted[column] = None
            elif column == "is_regimen":
                converted[column] = bool(value)
            else:
                converted[column] = str(value)
        records.append(converted)
    spark.createDataFrame(records, schema=schema).write.mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(_table_name(table))
    spark.sql(
        f"COMMENT ON TABLE {_table_name(table)} IS "
        + _sql_literal("Pharos CDM 2.0.1 reference lookup generated from pharos_cdm.")
    )
    print(f"published {TARGET}.{table}: {len(frame)} rows")


In [0]:

import json
from datetime import datetime, timezone
from pathlib import Path

catalog, schema_name = TARGET.split(".")
report_dir = Path(f"/Volumes/{catalog}/{schema_name}/artifacts/reports")
report_dir.mkdir(parents=True, exist_ok=True)
run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
if prof is not None:
    profile_path = report_dir / f"profile_seed{SEED}_{run_timestamp}.json"
    profile_path.write_text(json.dumps(prof, indent=2, sort_keys=True))
    print("wrote", profile_path)
run_report = {
    "utc_timestamp": run_timestamp,
    "mode": MODE,
    "seed": SEED,
    "persons": PERSONS,
    "source": SOURCE or None,
    "target": TARGET,
    "row_counts": ROW_COUNTS,
    "reference_row_counts": {table: int(len(frame)) for table, frame in REF_TABLES.items()},
    "validation": validation_report,
    "profile_coverage": PROFILE_COVERAGE,
    "mostly_ai": MOSTLY_REPORT,
    "pharos_cdm": CDM_SOURCE_INFO,
    "generator_code": CODE_SOURCE_INFO,
}
run_path = report_dir / f"run_seed{SEED}_{run_timestamp}.json"
run_path.write_text(json.dumps(run_report, indent=2, sort_keys=True))
print("wrote", run_path)

all_tables = sorted(list(ds.tables) + list(REF_TABLES))
count_sql = " UNION ALL ".join(
    f"SELECT '{table}' AS table_name, count(*) AS row_count FROM {_table_name(table)}"
    for table in all_tables
)
display(spark.sql(count_sql + " ORDER BY table_name"))